# Google Searches for "F1" or "Formula 1" Over Time

In [1]:
import pandas as pd
import altair as alt

## Get Google Search Data

In [2]:
f1_popularity_us_df = pd.read_csv("../../data/Formula_1_Search_Popularity_US.csv", skiprows = 1)
f1_popularity_worldwide_df = pd.read_csv("../../data/Formula_1_Search_Popularity_Worldwide.csv", skiprows = 1)

# Clean and rename columns
f1_popularity_us_df.columns = ["Month", "United States"]
f1_popularity_worldwide_df.columns = ["Month", "Worldwide"]

# Convert Month to datetime format
f1_popularity_us_df["Month"] = pd.to_datetime(f1_popularity_us_df["Month"])
f1_popularity_worldwide_df["Month"] = pd.to_datetime(f1_popularity_worldwide_df["Month"])

# Merge on Month
merged_df = pd.merge(f1_popularity_us_df, f1_popularity_worldwide_df, on = "Month")

In [3]:
merged_df.head()

,Month,United States,Worldwide
0,2015-01-01,11,15
1,2015-02-01,13,22
2,2015-03-01,20,40
3,2015-04-01,17,31
4,2015-05-01,18,29


## Create Plot

In [4]:
# Melt for Altair
long_df = merged_df.melt(id_vars = "Month", var_name = "Region", value_name = "Search Interest")

# mouseover nearest point
nearest = alt.selection(type = "single", 
                        nearest = True, 
                        on = "mouseover", 
                        fields = ["Month"], 
                        empty = "none")

# Base line chart
line = alt.Chart(long_df).mark_line().encode(
    x = alt.X("Month:T", title = "Date", axis = alt.Axis(grid = False)),
    y = alt.Y("Search Interest:Q", title = "Relative Google Search Popularity", axis = alt.Axis(grid = False)),
    color = alt.Color(
        "Region:N",
        title = "Region",
        scale = alt.Scale(range = ["#FF1E00", "#15151E"])
    )
)

# Transparent selectors along the x-axis
selectors = alt.Chart(long_df).mark_point().encode(
    x = "Month:T",
    opacity = alt.value(0),
).add_selection(
    nearest
)


# Tooltip box on hover
tooltips = alt.Chart(long_df).mark_circle(size = 200, opacity = 1).encode(
    x = "Month:T",
    y = "Search Interest:Q",
    color = "Region:N",
    tooltip = [
        alt.Tooltip("Month:T", title = "Month"),
        alt.Tooltip("Region:N", title = "Region"),
        alt.Tooltip("Search Interest:Q", title = "Search Popularity")
    ]
).transform_filter(
    nearest
)

# Vertical rule on hover
rule = alt.Chart(long_df).mark_rule(color = "black").encode(
    x = "Month:T"
).transform_filter(
    nearest
)

# Vertical line for Drive to Survive debut
debut_line = alt.Chart(pd.DataFrame({
    "Month": [pd.to_datetime("2019-03-01")],
    "label": ["Drive to Survive debuts"]
})).mark_rule(color = "black", strokeDash = [5, 5]).encode(
    x = "Month:T"
)

# Text annotation for debut
debut_text = alt.Chart(pd.DataFrame({
    "Month": [pd.to_datetime("2019-03-01")],
    "y": [long_df["Search Interest"].max() * .75],
    "label": ["F1: Drive to Survive debuts"]
})).mark_text(
    align = "left",
    baseline = "top",
    dx = 5,
    dy = -10,
    fontSize = 14,
    color = "black",
    font = "Quicksand, sans-serif"
).encode(
    x = "Month:T",
    y = alt.Y("y:Q"),
    text = "label:N"
)

# Combine all layers
chart = alt.layer(
    debut_line, debut_text, line, selectors, tooltips, rule
).properties(
    width = 750,
    height = 400,
    title = "Formula 1 Relative Search Popularity"
)

chart = chart.configure(
    title = {
        "font": "Quicksand, sans-serif", 
        "fontSize": 20
    },
    axis = {
        "labelFont": "Quicksand, sans-serif", 
        "titleFont": "Quicksand, sans-serif",
        "labelFontSize": 16,
        "titleFontSize": 16
    },
    legend = {
        "labelFont": "Quicksand, sans-serif",
        "titleFont": "Quicksand, sans-serif",
        "labelFontSize": 16,
        "titleFontSize": 16
    }
)

#chart.show()


C:\Users\DCorc\AppData\Local\Temp\ipykernel_26580\3854986117.py:5: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use 'selection_point()' or 'selection_interval()' instead.
These functions also include more helpful docstrings.
  nearest = alt.selection(type = "single",
C:\Users\DCorc\AppData\Local\Temp\ipykernel_26580\3854986117.py:26: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


In [ ]:
#chart.save("../../website/img/google_searches_over_time.html")

Numbers represent search interest relative to the highest point on the chart for the given region and time. A value of 100 is the peak popularity for the term. A value of 50 means that the term is half as popular. A score of 0 means there was not enough data for this term.